# Notebook 04 — SAM/BAM Format Deep Dive

**Module:** 09 — Genomics and NGS  
**Notebook:** 04 of 16  
**Time estimate:** 90 minutes

> **Track A critical:** The SAM format is the universal exchange format of NGS.
> Know the 11 mandatory fields, FLAG bit arithmetic, and CIGAR parsing cold.

---
## Step 1 — Motivation

Every aligner (BWA, STAR, HISAT2, minimap2) produces SAM/BAM output. Every
downstream tool (GATK, samtools, featureCounts, IGV) reads SAM/BAM. You cannot
work in NGS without understanding this format at the byte level.

---
## Step 2 — Intuition

SAM (Sequence Alignment/Map) is a tab-delimited text format. Each line after the
header describes one read alignment: where it mapped, how well, and what the
sequence looks like relative to the reference.

BAM is the binary, compressed version of SAM — same information, ~5× smaller.
CRAM is a reference-compressed version (~3× smaller than BAM).

The 11 mandatory fields in every SAM line describe:
1. Read name
2. Bit flags (tells you: paired, mapped, reverse strand, etc.)
3. Reference chromosome
4. Alignment start position
5. Mapping quality
6. CIGAR string (alignment operations)
7. Mate chromosome
8. Mate position
9. Insert size
10. Read sequence
11. Quality string

---
## Step 3 — Biological Background

**SAM flags encode biological/technical read properties as bit flags:**

| Bit | Decimal | Meaning |
|-----|---------|--------|
| 0x1 | 1 | Read is paired |
| 0x2 | 2 | Both mates mapped in proper pair |
| 0x4 | 4 | Read unmapped |
| 0x8 | 8 | Mate unmapped |
| 0x10 | 16 | Read maps to reverse strand |
| 0x20 | 32 | Mate maps to reverse strand |
| 0x40 | 64 | Read is mate 1 (read 1 of pair) |
| 0x80 | 128 | Read is mate 2 (read 2 of pair) |
| 0x100 | 256 | Secondary alignment |
| 0x200 | 512 | Read fails quality check |
| 0x400 | 1024 | PCR or optical duplicate |
| 0x800 | 2048 | Supplementary alignment (chimeric) |

**Decoding FLAG 99:** 64+32+2+1 = 99 → paired, properly mapped, mate on reverse
strand, read 1.

**CIGAR operations:**
- **M** — alignment match (can be match or mismatch)
- **I** — insertion to reference (bases in read, not in reference)
- **D** — deletion from reference (bases in reference, not in read)
- **S** — soft clipping (bases in read, not aligned)
- **H** — hard clipping (bases not in read sequence)
- **N** — skipped region from reference (intron in RNA-seq)
- **=** — exact match; **X** — mismatch

---
## Step 4 — Mathematical Explanation

**CIGAR consumed bases:**

| Op | Consumes query | Consumes reference |
|----|---------------|-------------------|
| M | Yes | Yes |
| I | Yes | No |
| D | No | Yes |
| S | Yes | No |
| H | No | No |
| N | No | Yes |

**Reference end position from CIGAR:**
$$\text{ref\_end} = \text{POS} + \sum_{\text{ops consuming ref}} \text{length} - 1$$

**Query length from CIGAR:**
$$\text{query\_len} = \sum_{\text{ops consuming query}} \text{length}$$

**Insert size (TLEN):** For paired-end reads, TLEN is the distance from the leftmost
base of mate1 to the rightmost base of mate2, including both mates. Positive for the
leftmost mate, negative for the rightmost.

In [ ]:
# Step 6 — Python: SAM parser from scratch
import re
from dataclasses import dataclass, field
from typing import Iterator


@dataclass
class CigarOp:
    length: int
    op: str

    # Which operations consume query (read) or reference
    CONSUMES_QUERY = set('MISH=X')
    CONSUMES_REF = set('MDNX=')

    @property
    def consumes_query(self) -> bool:
        return self.op in self.CONSUMES_QUERY

    @property
    def consumes_ref(self) -> bool:
        return self.op in self.CONSUMES_REF


def parse_cigar(cigar: str) -> list[CigarOp]:
    """Parse CIGAR string like '76M2I10M3D5S' into list of CigarOp."""
    if cigar == '*':
        return []
    ops = []
    for length, op in re.findall(r'(\d+)([MIDNSHP=X])', cigar):
        ops.append(CigarOp(int(length), op))
    return ops


def cigar_ref_length(cigar_ops: list[CigarOp]) -> int:
    return sum(op.length for op in cigar_ops if op.consumes_ref)


def cigar_query_length(cigar_ops: list[CigarOp]) -> int:
    return sum(op.length for op in cigar_ops if op.consumes_query)


def decode_flag(flag: int) -> dict:
    """Decode SAM FLAG integer into named boolean properties."""
    flag_bits = {
        'paired':             0x001,
        'proper_pair':        0x002,
        'read_unmapped':      0x004,
        'mate_unmapped':      0x008,
        'read_reverse':       0x010,
        'mate_reverse':       0x020,
        'read1':              0x040,
        'read2':              0x080,
        'secondary':          0x100,
        'qc_fail':            0x200,
        'duplicate':          0x400,
        'supplementary':      0x800,
    }
    return {name: bool(flag & bit) for name, bit in flag_bits.items()}


@dataclass
class SamRecord:
    qname: str
    flag: int
    rname: str
    pos: int          # 1-based
    mapq: int
    cigar: str
    rnext: str
    pnext: int
    tlen: int
    seq: str
    qual: str
    tags: dict = field(default_factory=dict)

    @property
    def cigar_ops(self) -> list[CigarOp]:
        return parse_cigar(self.cigar)

    @property
    def ref_end(self) -> int:
        return self.pos + cigar_ref_length(self.cigar_ops) - 1

    @property
    def is_mapped(self) -> bool:
        return not (self.flag & 0x4)

    @property
    def is_duplicate(self) -> bool:
        return bool(self.flag & 0x400)

    @property
    def on_reverse_strand(self) -> bool:
        return bool(self.flag & 0x10)


def parse_sam_line(line: str) -> SamRecord | None:
    """Parse one SAM alignment line."""
    if line.startswith('@'):
        return None
    fields = line.rstrip('\n').split('\t')
    if len(fields) < 11:
        return None

    # Parse optional tags (key:type:value)
    tags = {}
    for tag_str in fields[11:]:
        parts = tag_str.split(':', 2)
        if len(parts) == 3:
            key, typ, val = parts
            if typ in ('i', 'I'):
                val = int(val)
            elif typ == 'f':
                val = float(val)
            tags[key] = val

    return SamRecord(
        qname=fields[0],
        flag=int(fields[1]),
        rname=fields[2],
        pos=int(fields[3]),
        mapq=int(fields[4]),
        cigar=fields[5],
        rnext=fields[6],
        pnext=int(fields[7]),
        tlen=int(fields[8]),
        seq=fields[9],
        qual=fields[10],
        tags=tags,
    )


# Test
test_lines = [
    'read001\t99\tchr1\t1000\t60\t76M\t=\t1200\t276\tACGTACGTACGT\t!!!!!!!!!!!!\tNM:i:0\tAS:i:76',
    'read001\t147\tchr1\t1200\t60\t76M\t=\t1000\t-276\tGCATGCATGCAT\t!!!!!!!!!!!!\tNM:i:1',
    'read002\t4\t*\t0\t0\t*\t*\t0\t0\tACGTACGT\t!!!!!!!!',
    'read003\t0\tchr1\t5000\t30\t10M3I5M2D20M\t*\t0\t0\tACGTACGTACGTACGTACGTACGTACGTACGTACGT\t!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!',
]

for line in test_lines:
    r = parse_sam_line(line)
    if r is None:
        continue
    flags = decode_flag(r.flag)
    ops = r.cigar_ops
    print(f"\n--- {r.qname} ---")
    print(f"  FLAG={r.flag}: paired={flags['paired']}, mapped={r.is_mapped}, "
          f"reverse={r.on_reverse_strand}, duplicate={r.is_duplicate}")
    print(f"  Position: {r.rname}:{r.pos}-{r.ref_end} (MAPQ={r.mapq})")
    print(f"  CIGAR: {r.cigar} → ops={[(o.length, o.op) for o in ops]}")
    print(f"  Query length: {cigar_query_length(ops)}, Ref consumed: {cigar_ref_length(ops)}")
    print(f"  Tags: {r.tags}")

In [ ]:
# CIGAR-based alignment reconstruction
def reconstruct_alignment_from_cigar(
    read_seq: str,
    cigar: str,
    ref_seq: str,
    ref_start: int = 0
) -> tuple[str, str, str]:
    """
    Reconstruct aligned read and reference strings from CIGAR.
    Returns (aligned_read, alignment_bar, aligned_ref).
    """
    ops = parse_cigar(cigar)
    read_str, ref_str, bar_str = [], [], []
    q_pos = 0  # position in read
    r_pos = ref_start  # position in reference

    for op in ops:
        n = op.length
        if op.op == 'M' or op.op == '=' or op.op == 'X':
            for _ in range(n):
                rb = ref_seq[r_pos] if r_pos < len(ref_seq) else 'N'
                qb = read_seq[q_pos] if q_pos < len(read_seq) else 'N'
                read_str.append(qb)
                ref_str.append(rb)
                bar_str.append('|' if qb == rb else 'X')
                q_pos += 1
                r_pos += 1
        elif op.op == 'I':
            for _ in range(n):
                read_str.append(read_seq[q_pos])
                ref_str.append('-')
                bar_str.append('I')
                q_pos += 1
        elif op.op == 'D' or op.op == 'N':
            for _ in range(n):
                read_str.append('-')
                rb = ref_seq[r_pos] if r_pos < len(ref_seq) else 'N'
                ref_str.append(rb)
                bar_str.append('D' if op.op == 'D' else 'N')
                r_pos += 1
        elif op.op == 'S':
            q_pos += n  # soft-clipped: skip read, nothing in ref

    return ''.join(read_str), ''.join(bar_str), ''.join(ref_str)


# Example: read with insertion and deletion
ref   = 'ATCGATCGATCGATCGATCGATCG'
read  = 'ATCGAAATCGATCGATG'  # insertion of AAA, deletion of ATCG at end
cigar = '4M3I6M4D'  # example

aligned_read, bar, aligned_ref = reconstruct_alignment_from_cigar(read, cigar, ref)

print("CIGAR-based alignment reconstruction:")
print(f"CIGAR:   {cigar}")
print(f"Read:    {aligned_read}")
print(f"Bar:     {bar}")
print(f"Ref:     {aligned_ref}")

# Flag decoding table
print("\nFLAG decoding examples:")
for flag_val, description in [
    (99, 'paired-end read 1, properly paired, mate on reverse strand'),
    (147, 'paired-end read 2, properly paired, this read on reverse strand'),
    (4, 'unmapped read'),
    (1024, 'PCR duplicate (will be filtered by GATK)'),
    (256, 'secondary alignment (not the best alignment for this read)'),
    (2048, 'supplementary alignment (chimeric)'),
]:
    decoded = decode_flag(flag_val)
    true_flags = [k for k, v in decoded.items() if v]
    print(f"  FLAG={flag_val:4d}: {', '.join(true_flags)}")
    print(f"         (description: {description})")

In [ ]:
# Step 7 — Visualization: SAM format anatomy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel A: SAM line anatomy
ax = axes[0, 0]
ax.axis('off')
fields_info = [
    ('QNAME', 'read001', '#4FC3F7'),
    ('FLAG', '99', '#81C784'),
    ('RNAME', 'chr1', '#FFB74D'),
    ('POS', '1000', '#F06292'),
    ('MAPQ', '60', '#CE93D8'),
    ('CIGAR', '76M', '#80CBC4'),
    ('RNEXT', '=', '#4FC3F7'),
    ('PNEXT', '1200', '#81C784'),
    ('TLEN', '276', '#FFB74D'),
    ('SEQ', 'ACGT...', '#F06292'),
    ('QUAL', 'IIII...', '#CE93D8'),
]
for i, (name, val, color) in enumerate(fields_info):
    x = i / len(fields_info)
    ax.add_patch(mpatches.FancyBboxPatch(
        (x + 0.005, 0.55), 1/len(fields_info) - 0.01, 0.3,
        boxstyle='round,pad=0.01', facecolor=color, edgecolor='gray', lw=1
    ))
    ax.text(x + 0.5/len(fields_info), 0.70, name, ha='center', va='center', fontsize=7, fontweight='bold')
    ax.text(x + 0.5/len(fields_info), 0.60, val, ha='center', va='center', fontsize=7)
ax.set_title('A. The 11 mandatory SAM fields', fontsize=11, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Panel B: FLAG bit table
ax2 = axes[0, 1]
ax2.axis('off')
flag_data = [
    ['0x001 (1)', 'Paired'],
    ['0x002 (2)', 'Proper pair'],
    ['0x004 (4)', 'Read unmapped'],
    ['0x008 (8)', 'Mate unmapped'],
    ['0x010 (16)', 'Read reverse'],
    ['0x020 (32)', 'Mate reverse'],
    ['0x040 (64)', 'Read 1'],
    ['0x080 (128)', 'Read 2'],
    ['0x100 (256)', 'Secondary aln'],
    ['0x200 (512)', 'QC fail'],
    ['0x400 (1024)', 'Duplicate'],
    ['0x800 (2048)', 'Supplementary'],
]
col_labels = ['Bit (decimal)', 'Meaning']
t = ax2.table(flag_data, colLabels=col_labels, loc='center', cellLoc='left')
t.auto_set_font_size(False)
t.set_fontsize(9)
t.scale(1, 1.2)
ax2.set_title('B. SAM FLAG bit definitions', fontsize=11, fontweight='bold')

# Panel C: CIGAR operations
ax3 = axes[1, 0]
# Draw a visual CIGAR alignment: 5M2I3M4D2M
ax3.axis('off')
ref_row  = list('ATCGATCG....ATCG')
read_row = list('ATCGAACGATCG....')
bar_row  = ['|','|','|','|','I','I','|','|','|','|','D','D','D','D','|','|']
cigar_label = '4M  2I  4M  4D  2M'

for i, (r, b, q) in enumerate(zip(ref_row, bar_row, read_row)):
    color = {'|': '#81C784', 'I': '#FFB74D', 'D': '#F06292', 'X': '#EF5350'}.get(b, 'white')
    ax3.add_patch(mpatches.FancyBboxPatch(
        (i * 0.058 + 0.01, 0.55), 0.05, 0.12,
        boxstyle='round,pad=0.005', facecolor=color, edgecolor='gray', lw=0.5, alpha=0.7
    ))
    ax3.text(i * 0.058 + 0.035, 0.68, q, ha='center', va='center', fontsize=8)
    ax3.text(i * 0.058 + 0.035, 0.61, b, ha='center', va='center', fontsize=6)
    ax3.text(i * 0.058 + 0.035, 0.48, r, ha='center', va='center', fontsize=8)

ax3.text(0.5, 0.80, 'Read', ha='center', fontsize=9, fontweight='bold')
ax3.text(0.5, 0.42, 'Reference', ha='center', fontsize=9, fontweight='bold')
ax3.text(0.5, 0.25, f'CIGAR: {cigar_label}', ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='#E3F2FD', edgecolor='steelblue'))
legend_handles = [
    mpatches.Patch(color='#81C784', label='M: match/mismatch'),
    mpatches.Patch(color='#FFB74D', label='I: insertion'),
    mpatches.Patch(color='#F06292', label='D: deletion'),
]
ax3.legend(handles=legend_handles, loc='lower right', fontsize=7)
ax3.set_title('C. CIGAR string: visual interpretation', fontsize=11, fontweight='bold')
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)

# Panel D: FLAG decode example
ax4 = axes[1, 1]
ax4.axis('off')
example_flags = [99, 147, 4, 1024, 256, 2048]
flag_names = []
for f in example_flags:
    d = decode_flag(f)
    true_bits = [k for k, v in d.items() if v]
    flag_names.append((f, ', '.join(true_bits[:3]) + ('...' if len(true_bits) > 3 else '')))

table_data = [[str(f), desc] for f, desc in flag_names]
t4 = ax4.table(table_data, colLabels=['FLAG', 'Decoded meaning'], loc='center', cellLoc='left')
t4.auto_set_font_size(False)
t4.set_fontsize(9)
t4.scale(1, 1.8)
ax4.set_title('D. Common FLAG values decoded', fontsize=11, fontweight='bold')

plt.suptitle('SAM/BAM Format Reference', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sam_bam_format.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Write a function `flag_to_int(flags: dict) -> int` that converts a dict of flag
   names to a FLAG integer. Verify: `{'paired': True, 'read1': True, 'proper_pair': True, 'mate_reverse': True}` → 99.
2. Parse CIGAR `'20M5I15M3D10M'`. How many bases does the read span? How many
   bases does it span on the reference?
3. A read has FLAG=147. Decode it: is it read 1 or read 2? Is it mapped? Which
   strand does it map to?
4. Write a function that, given a SAM file (as a string of lines), returns the
   fraction of reads that are: (a) unmapped, (b) duplicates, (c) secondary alignments.

---
## Step 10 — Quiz

1. What are the 11 mandatory SAM fields?
2. What does CIGAR operation 'N' mean? How is it different from 'D'?
3. A read has FLAG bit 0x4 set. What does this mean?
4. How do you compute the reference end position from POS and CIGAR?
5. What is the difference between a secondary (FLAG 256) and supplementary (FLAG 2048) alignment?

---
## Step 12 — Reflection

> *[In one paragraph: explain what information you lose when converting a SAM file
> to a BAM file, and what information you lose when further converting to CRAM.]*

---
*Next: `05_alignment_statistics_mapq_coverage.ipynb`*